# Build STDN Context Features

This notebook aligns holiday and weather data to 30-minute bins, fills missing values safely, and exports additive context features centered around 0.0.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

BASE_DIR = Path('MockData') if (Path('MockData') / 'holidays_2025.csv').exists() else Path('.')
HOLIDAY_PATH = BASE_DIR / 'holidays_2025.csv'
WEATHER_PATH = BASE_DIR / 'weather_2025.csv'

YEAR = 2025
FREQ = '30min'

OUT_CSV = BASE_DIR / 'context_2025.csv'
OUT_NPZ = BASE_DIR / 'context_2025.npz'
OUT_META = BASE_DIR / 'context_2025_meta.json'

In [2]:
# Small, conservative additive signals. A normal slot stays at 0.0.
EVENT_WEIGHTS = {
    'Is_Legal_Holiday': 0.10,
    'Is_School_Recess': 0.05,
    'Is_Event_Festival': 0.08,
    'Is_Early_Dismissal': 0.04,
}

# Weather features are already normalized to [0, 1], so center them at 0.5.
# This keeps weather centered at 0 instead of shifting every slot upward.
TEMP_ALPHA = 0.10
CLOUD_ALPHA = 0.05
WIND_ALPHA = 0.05

# Rain one-hot is very sparse for moderate/heavy rain, so export one compact
# intensity feature instead of three mostly-zero columns.
RAIN_WEIGHTS = {
    'Rain_Light': 0.03,
    'Rain_Moderate': 0.08,
    'Rain_Heavy': 0.15,
}

def full_time_index(year, freq='30min'):
    start = pd.Timestamp(f'{year}-01-01 00:00:00')
    end = pd.Timestamp(f'{year}-12-31 23:30:00')
    return pd.date_range(start, end, freq=freq)

def align_to_timebins(df, full_index, agg, full_day_defaults):
    df = df.copy()
    df['time'] = pd.to_datetime(df['time'], errors='coerce').dt.floor(FREQ)
    df = df.dropna(subset=['time'])

    value_cols = [c for c in df.columns if c != 'time']
    grouped = df.groupby('time', as_index=True)[value_cols].agg(agg)

    all_days = pd.Index(full_index.normalize().unique())
    present_days = pd.Index(grouped.index.normalize().unique())
    full_missing_days = all_days.difference(present_days)

    aligned = grouped.reindex(full_index)

    # Missing single bins use the nearest previous bin; leading gaps fall back
    # to the next available bin because no previous slot exists yet.
    aligned = aligned.ffill().bfill()

    # Missing whole days should not inherit neighboring days. They are neutral.
    full_day_mask = aligned.index.normalize().isin(full_missing_days)
    for col, value in full_day_defaults.items():
        aligned.loc[full_day_mask, col] = value

    return aligned, list(full_missing_days)

def build_context_features(holiday_path, weather_path, year=2024):
    full_index = full_time_index(year, FREQ)

    holiday_raw = pd.read_csv(holiday_path)
    weather_raw = pd.read_csv(weather_path)

    holiday_defaults = {col: 0 for col in holiday_raw.columns if col != 'time'}
    weather_defaults = {
        'temperature_2m': 0.5,
        'cloudcover': 0.5,
        'windspeed_10m': 0.5,
        'Rain_None': 1,
        'Rain_Light': 0,
        'Rain_Moderate': 0,
        'Rain_Heavy': 0,
    }

    holiday, missing_holiday_days = align_to_timebins(
        holiday_raw, full_index, agg='max', full_day_defaults=holiday_defaults
    )
    weather, missing_weather_days = align_to_timebins(
        weather_raw, full_index, agg='mean', full_day_defaults=weather_defaults
    )

    context = pd.DataFrame(index=full_index)

    for col, weight in EVENT_WEIGHTS.items():
        context[col.replace('Is_', '') + '_effect'] = holiday[col].astype(float).clip(0, 1) * weight

    context['temperature_effect'] = TEMP_ALPHA * (weather['temperature_2m'].astype(float).clip(0, 1) - 0.5)
    context['cloudcover_effect'] = CLOUD_ALPHA * (weather['cloudcover'].astype(float).clip(0, 1) - 0.5)
    context['windspeed_effect'] = WIND_ALPHA * (weather['windspeed_10m'].astype(float).clip(0, 1) - 0.5)

    rain_score = np.zeros(len(weather), dtype=np.float32)
    for col, weight in RAIN_WEIGHTS.items():
        rain_score += weather[col].astype(float).clip(0, 1).to_numpy(dtype=np.float32) * weight
    context['rain_intensity_effect'] = rain_score

    # If a whole day is missing in either source, keep all context features neutral.
    missing_days = pd.Index(missing_holiday_days).union(pd.Index(missing_weather_days))
    full_day_mask = context.index.normalize().isin(missing_days)
    context.loc[full_day_mask, :] = 0.0

    context.insert(0, 'time', context.index.strftime('%Y-%m-%d %H:%M:%S'))
    return context.reset_index(drop=True), missing_holiday_days, missing_weather_days

context_df, missing_holiday_days, missing_weather_days = build_context_features(
    HOLIDAY_PATH, WEATHER_PATH, YEAR
)

print(context_df.shape)
print(context_df.head(10))
print('Missing full holiday days:', missing_holiday_days[:10], 'count=', len(missing_holiday_days))
print('Missing full weather days:', missing_weather_days[:10], 'count=', len(missing_weather_days))

(17520, 9)
                  time  Legal_Holiday_effect  School_Recess_effect  \
0  2025-01-01 00:00:00                   0.1                  0.05   
1  2025-01-01 00:30:00                   0.1                  0.05   
2  2025-01-01 01:00:00                   0.1                  0.05   
3  2025-01-01 01:30:00                   0.1                  0.05   
4  2025-01-01 02:00:00                   0.1                  0.05   
5  2025-01-01 02:30:00                   0.1                  0.05   
6  2025-01-01 03:00:00                   0.1                  0.05   
7  2025-01-01 03:30:00                   0.1                  0.05   
8  2025-01-01 04:00:00                   0.1                  0.05   
9  2025-01-01 04:30:00                   0.1                  0.05   

   Event_Festival_effect  Early_Dismissal_effect  temperature_effect  \
0                    0.0                     0.0           -0.006705   
1                    0.0                     0.0           -0.006705   
2 

In [3]:
feature_cols = [c for c in context_df.columns if c != 'time']
context_array = context_df[feature_cols].to_numpy(dtype=np.float32)

context_df.to_csv(OUT_CSV, index=False)
np.savez_compressed(
    OUT_NPZ,
    context=context_array,
    time=context_df['time'].to_numpy(),
    feature_names=np.array(feature_cols),
)

meta = {
    'year': YEAR,
    'freq': FREQ,
    'shape': list(context_array.shape),
    'feature_names': feature_cols,
    'missing_full_holiday_days': [str(d.date()) for d in missing_holiday_days],
    'missing_full_weather_days': [str(d.date()) for d in missing_weather_days],
    'neutral_value': 0.0,
    'rain_handling': 'Rain one-hot columns are compressed into rain_intensity_effect.',
}
OUT_META.write_text(json.dumps(meta, indent=2), encoding='utf-8')

print('Saved:', OUT_CSV)
print('Saved:', OUT_NPZ)
print('Saved:', OUT_META)
print(context_df.describe().T)

Saved: context_2025.csv
Saved: context_2025.npz
Saved: context_2025_meta.json
                          count      mean       std    min       25%  \
Legal_Holiday_effect    17520.0  0.003288  0.017832  0.000  0.000000   
School_Recess_effect    17520.0  0.003699  0.013087  0.000  0.000000   
Event_Festival_effect   17520.0  0.001753  0.011714  0.000  0.000000   
Early_Dismissal_effect  17520.0  0.000658  0.005086  0.000  0.000000   
temperature_effect      17520.0  0.001149  0.018773 -0.050 -0.014330   
cloudcover_effect       17520.0  0.003850  0.021062 -0.025 -0.021000   
windspeed_effect        17520.0 -0.008783  0.008638 -0.025 -0.015376   
rain_intensity_effect   17520.0  0.005298  0.013949  0.000  0.000000   

                             50%       75%    max  
Legal_Holiday_effect    0.000000  0.000000  0.100  
School_Recess_effect    0.000000  0.000000  0.050  
Event_Festival_effect   0.000000  0.000000  0.080  
Early_Dismissal_effect  0.000000  0.000000  0.040  
temperature_e

In [4]:
# Quick sanity checks
expected_slots = len(full_time_index(YEAR, FREQ))
assert len(context_df) == expected_slots, (len(context_df), expected_slots)
assert np.isfinite(context_array).all()
assert np.abs(context_array).max() <= 1.0

print('Expected slots:', expected_slots)
print('Context range:', float(context_array.min()), float(context_array.max()))
print('Feature columns:', feature_cols)

Expected slots: 17520
Context range: -0.05000000074505806 0.15000000596046448
Feature columns: ['Legal_Holiday_effect', 'School_Recess_effect', 'Event_Festival_effect', 'Early_Dismissal_effect', 'temperature_effect', 'cloudcover_effect', 'windspeed_effect', 'rain_intensity_effect']
